# Environment Setup & VRAM Memory Manager

In [1]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate tqdm

import os
import sys
import json
import gc
import time
import random
import torch
import transformers, peft, trl
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# Directories for Kaggle output
OUTPUT_DIR = "/kaggle/working/staged_decoding"
DATA_DIR = "/kaggle/working/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Helper function to flush PyTorch CUDA cache between training stages
def clear_gpu_memory(model_obj=None, trainer_obj=None):
    if trainer_obj is not None:
        del trainer_obj
    if model_obj is not None:
        del model_obj
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        free_mem, total_mem = torch.cuda.mem_get_info()
        print(f"VRAM Cleared! Free GPU Memory: {free_mem / 1e9:.2f} GB / {total_mem / 1e9:.2f} GB")

print("Setup complete and VRAM manager initialized.")

Setup complete and VRAM manager initialized.


In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ Hugging Face authenticated.")

✅ Hugging Face authenticated.


# Re-balanced Dataset Generation (50% Control Split)

In [3]:
def generate_staged_datasets(n_total=500):
    stage1_file = os.path.join(DATA_DIR, "stage1_privacy_dataset.jsonl")
    stage2_file = os.path.join(DATA_DIR, "stage2_accuracy_dataset.jsonl")

    vars_pool = ['A', 'B', 'C', 'D', 'X', 'Y', 'Z']

    n_type_c = n_total // 2
    n_type_a = n_total // 4
    n_type_b = n_total // 4

    stage1_samples = []
    stage2_samples = []

    # Strict system prompt: no reasoning allowed in Stage 1's output channel.
    STAGE1_SYSTEM = (
        "You are a privacy-aware variable access filter. "
        "Respond with EXACTLY ONE LINE and NOTHING ELSE.\n"
        "Format: comma-separated get_operand(\"VAR\") calls for ONLY the "
        "necessary variables, e.g. get_operand(\"C\"), get_operand(\"D\")\n"
        "If no variables are necessary, respond with exactly: NONE\n"
        "Do NOT explain. Do NOT reason. Do NOT add any other text."
    )

    def add_entry(expr, data_str, keep_vars, answer, think_reasoning):
        calls_line = (
            ", ".join([f'get_operand("{v}")' for v in keep_vars])
            if keep_vars else "NONE"
        )

        s1_text = (
            f"<|start_header_id|>system<|end_header_id|>\n"
            f"{STAGE1_SYSTEM}\n"
            f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
            f"Expression: {expr}\nData: {data_str}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"{calls_line}<|eot_id|>"
        )
        stage1_samples.append({"text": s1_text})

        # Stage 2 
        s2_text = (
            f"<|start_header_id|>system<|end_header_id|>\n"
            f"You are a mathematical execution engine. Compute the final answer using ONLY the permitted variables.\n"
            f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
            f"Expression: {expr}\nPermitted Variables: {', '.join(keep_vars) if keep_vars else 'NONE'}\nData: {data_str}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"<think> {think_reasoning} </think> The final answer is {answer}.<|eot_id|>"
        )
        stage2_samples.append({"text": s2_text})

    # TYPE C: CONTROL (50%)
    for _ in range(n_type_c):
        vA, vB, varC = random.sample(vars_pool, 3)
        valA, valB, valC = random.randint(2, 50), random.randint(2, 50), random.randint(2, 50)
        expr = f"({vA} + {vB}) * {varC}"
        data = f"{vA}={valA}, {vB}={valB}, {varC}={valC}"
        ans = (valA + valB) * valC
        add_entry(expr, data, [vA, vB, varC], ans, f"All variables {vA}, {vB}, {varC} are required.")

    # TYPE A: GLOBAL TRAP (25%) 
    for _ in range(n_type_a):
        vA, vB, varC = random.sample(vars_pool, 3)
        valA, valB, valC = random.randint(2, 50), random.randint(2, 50), random.randint(2, 50)
        expr = f"(({vA} + {vB} - {varC}) * 0)"
        data = f"{vA}={valA}, {vB}={valB}, {varC}={valC}"
        add_entry(expr, data, [], 0, "Global zero-multiplier nullifies expression.")

    # TYPE B: PARTIAL TRAP (25%)
    for _ in range(n_type_b):
        vA, vB, varC = random.sample(vars_pool, 3)
        valA, valB, valC = random.randint(2, 50), random.randint(2, 50), random.randint(2, 50)
        expr = f"(({vA} * {vB}) * 0) + {varC}"
        data = f"{vA}={valA}, {vB}={valB}, {varC}={valC}"
        add_entry(expr, data, [varC], valC, f"Shortcut (*0) nullifies {vA},{vB}. Only {varC} needed.")

    with open(stage1_file, "w") as f:
        for s in stage1_samples:
            f.write(json.dumps(s) + "\n")

    with open(stage2_file, "w") as f:
        for s in stage2_samples:
            f.write(json.dumps(s) + "\n")

    print(f"Generated {len(stage1_samples)} training samples at {DATA_DIR}")
    print(f"Stage 1 target format: strict single-line calls (e.g. 'get_operand(\"C\")' or 'NONE')")

generate_staged_datasets(n_total=500)

Generated 500 training samples at /kaggle/working/data
Stage 1 target format: strict single-line calls (e.g. 'get_operand("C")' or 'NONE')


In [ ]:

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)

#  Stage 1 Training: Privacy Filter Adapter

In [4]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Base Model
model_s1 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)
model_s1.config.pad_token_id = tokenizer.pad_token_id


model_s1 = prepare_model_for_kbit_training(model_s1, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_s1 = get_peft_model(model_s1, lora_config)
model_s1.enable_input_require_grads()

# Load Stage 1 Data
stage1_file = os.path.join(DATA_DIR, "stage1_privacy_dataset.jsonl")
dataset_s1 = load_dataset("json", data_files=stage1_file, split="train")

trainer_s1 = SFTTrainer(
    model=model_s1,
    train_dataset=dataset_s1,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir="/kaggle/working/checkpoints_s1",
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        num_train_epochs=2,
        bf16=True,
        optim="paged_adamw_8bit",
        logging_steps=20,
        save_strategy="epoch",
        max_length=128,
        report_to="none",
        loss_type="nll",
    )
)

print("Training Stage 1 (Privacy Adapter)...")
trainer_s1.train()

# Save Stage 1 Adapter
privacy_adapter_path = os.path.join(OUTPUT_DIR, "privacy_adapter_final")
trainer_s1.model.save_pretrained(privacy_adapter_path)
tokenizer.save_pretrained(privacy_adapter_path)
print(f"Stage 1 Adapter saved to: {privacy_adapter_path}")

# Purge Stage 1 from GPU VRAM
clear_gpu_memory(model_s1, trainer_s1)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Training Stage 1 (Privacy Adapter)...


Step,Training Loss
20,2.954284
40,0.987699
60,0.307615
80,0.233974
100,0.216665
120,0.214339


Stage 1 Adapter saved to: /kaggle/working/staged_decoding/privacy_adapter_final
VRAM Cleared! Free GPU Memory: 11.54 GB / 15.64 GB


# Stage 2 Training: Accuracy Execution Adapter

In [ ]:
# Load Fresh Base Model Instance for Stage 2
model_s2 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)
model_s2.config.pad_token_id = tokenizer.pad_token_id
model_s2 = get_peft_model(model_s2, lora_config)
model_s2.enable_input_require_grads()

# Load Stage 2 Data
stage2_file = os.path.join(DATA_DIR, "stage2_accuracy_dataset.jsonl")
dataset_s2 = load_dataset("json", data_files=stage2_file, split="train")

trainer_s2 = SFTTrainer(
    model=model_s2,
    train_dataset=dataset_s2,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir="/kaggle/working/checkpoints_s2",
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        num_train_epochs=2,
        bf16=True,
        optim="paged_adamw_8bit",
        logging_steps=20,
        save_strategy="epoch",
        max_length=256,
        report_to="none"
    )
)

print("Training Stage 2 (Accuracy Adapter)...")
trainer_s2.train()

# Save Stage 2 Adapter
accuracy_adapter_path = os.path.join(OUTPUT_DIR, "accuracy_adapter_final")
trainer_s2.model.save_pretrained(accuracy_adapter_path)
tokenizer.save_pretrained(accuracy_adapter_path)
print(f"Stage 2 Adapter saved to: {accuracy_adapter_path}")

# Purge Stage 2 from GPU VRAM
clear_gpu_memory(model_s2, trainer_s2)

# Benchmark Evaluation with Live Telemetry Logging

In [ ]:
# 1. Clone RAF repository to Kaggle workspace
raf_repo_dir = "/kaggle/working/ReasoningAuthenticationFramework-RAF-"
if not os.path.exists(raf_repo_dir):
    !git clone https://github.com/ringerH/ReasoningAuthenticationFramework-RAF-.git {raf_repo_dir}

sys.path.append(raf_repo_dir)
from staged_local_stub import LocalStagedDecodingStub

# 2. Load Base Model in Evaluation Mode
eval_base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)

# 3. Initialize Stub pointing to Kaggle Adapter Paths
stub = LocalStagedDecodingStub(
    base_model=eval_base_model,
    tokenizer=tokenizer,
    privacy_adapter_path=privacy_adapter_path,
    accuracy_adapter_path=accuracy_adapter_path
)

# 4. Load Test Benchmark
benchmark_file = os.path.join(raf_repo_dir, "data/test_sets/iccr_test_set.jsonl")
with open(benchmark_file, "r") as f:
    test_cases = [json.loads(line) for line in f]

telemetry_results = []
running_dms = {"A": [], "B": [], "C": []}

print(f"Starting Live Evaluation on {len(test_cases)} benchmark items...\n")
print(f"{'Item':<10} | {'Type':<6} | {'Accessed':<12} | {'Item DMS':<8} | {'Latency':<8} | {'Running DMS (A/B/C)':<20}")
print("-" * 75)

# 5. Live Inference Loop
for idx, item in enumerate(tqdm(test_cases, desc="Evaluating Benchmark")):
    p_id = item.get("id", f"item_{idx}")
    p_type = item.get("type", "B")
    
    start_t = time.time()
    telemetry_log, final_ans = stub.run_staged_inference(item)
    latency = time.time() - start_t
    
    dms_score = telemetry_log["dms"]
    running_dms[p_type].append(dms_score)
    
    telemetry_results.append({
        "problem_id": p_id,
        "telemetry": telemetry_log,
        "final_answer": final_ans,
        "latency_sec": round(latency, 2)
    })
    
    avg_a = sum(running_dms["A"]) / len(running_dms["A"]) if running_dms["A"] else 0.0
    avg_b = sum(running_dms["B"]) / len(running_dms["B"]) if running_dms["B"] else 0.0
    avg_c = sum(running_dms["C"]) / len(running_dms["C"]) if running_dms["C"] else 0.0
    
    # Print progress every 2 items
    if (idx + 1) % 2 == 0 or idx == 0 or (idx + 1) == len(test_cases):
        acc_str = ",".join(telemetry_log["accessed"]) if telemetry_log["accessed"] else "NONE"
        print(f"{p_id:<10} | {p_type:<6} | {acc_str:<12} | {dms_score:<8.2f} | {latency:<6.2f}s | A:{avg_a:.2f} B:{avg_b:.2f} C:{avg_c:.2f}")

# 6. Save Telemetry Artifact
final_log_file = os.path.join(OUTPUT_DIR, "staged_decoding_telemetry_final.jsonl")
with open(final_log_file, "w") as f:
    for entry in telemetry_results:
        f.write(json.dumps(entry) + "\n")

print(f"\nBenchmark complete! Standardized Oracle JSON logs saved to: {final_log_file}")
